In [ ]:
import csv

with open("data/上級英語IB_ポイント解説 - Unit_2.csv", encoding="utf-8") as f:
    rows = list(csv.reader(f))

print("行数：", len(rows))
for r in rows[:6]:
    print(r)

In [ ]:
header_idx = next(i for i, r in enumerate(rows) if r and r[0] == "Unit#")
header = rows[header_idx]
print("見出し：", header)

records = [dict(zip(header, r)) for r in rows[header_idx + 1:] if any(r)]
print(len(records), "件")
print(records[0])

In [ ]:
chunks = []
for rec in records:
    text = (
        f"Unit {rec.get('Unit#', '')} Lesson {rec.get('Lesson#', '')}\n"
        f"タイトル: {rec.get('Lesson Title', '')}\n"
        f"種類: {rec.get('Type', '')}\n"
        f"本文: {rec.get('Transcript', '')}\n"
        f"ポイント解説: {rec.get('ポイント解説', '')}"
    )
    chunks.append({
        "text": text,
        "source": "上級英語IB_ポイント解説",
        "unit": rec.get("Unit#", ""),
        "lesson": rec.get("Lesson#", ""),
    })

print(len(chunks), "個のチャンク")
print(chunks[0]["text"])

In [ ]:
from openai import OpenAI
import chromadb

client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")
EMBED_MODEL = "text-embedding-bge-m3"

def embed(text):
    response = client.embeddings.create(model=EMBED_MODEL, input=text)
    return response.data[0].embedding

db = chromadb.PersistentClient(path="chroma_db") # `chroma_db`フォルダにデータを保存
col = db.get_or_create_collection("englich", metadata={"hnsw:space": "cosine"}) # `collection`はDBの中の「棚」みたいなもの,`cosine`は似てる度をコサインで図る設定

In [ ]:
col.upsert(
    ids=[f"{c['source']}_{i}" for i, c in enumerate(chunks)],
    documents=[c["text"] for c in chunks],
    embeddings=[embed(c["text"]) for c in chunks],
    metadatas=[{"source": c["source"], "unit": c["unit"], "lesson": c["lesson"]} for c in chunks],
)

print("保存した件数：", col.count())

- `ids`:1件ずつの名札
- `documents`:本文
- `embeddings`:数字
- `metadatas`:出典情報
- `upsert`:同じ名札があれば上書き、なければ追加。

In [ ]:
q = "keep in touchってどうゆう意味？"

res = col.query(query_embeddings=[embed(q)], n_results=2)

for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
    print(f"似てる度:{1 - dist:.3f} Unit {meta['unit']} Lesson {meta['lesson']}")
    print(doc[:100], "...")
    print("-----")

In [ ]:
CHAT_MODEL = "qwen3.6-35b-a3b"

def ask(question, k=2):
    res = col.query(query_embeddings=[embed(question)], n_results=k)
    docs = res["documents"][0]
    metas = res["metadatas"][0]

    context = "\n---\n".join(docs)
    messages = [
        {"role": "system", "content": "資料だけを根拠に日本語で答えて。資料にないことは『資料に記載がありません』と答えて。"},
        {"role": "user", "content": f"資料:\n{context}\n\n質問: {question}"},
    ]
    response = client.chat.completions.create(model=CHAT_MODEL, messages=messages, temperature=0.2)

    answer = response.choices[0].message.content.strip()
    sources = [f"Unit {m['unit']} Lesson {m['lesson']}" for m in metas]
    return answer + "\n\n📚 参照：" + "、".join(sources)

print(ask("keep in touchってどういう意味？"))